# Run the Wallstreet-AI Pipeline in Google Colab

This notebook sets up the `wallstreet-ai` project in Google Colab and runs its main analysis pipeline with a single query.

The pipeline does more than call an LLM. It parses the user query, selects the required analysis flow, gathers market/news data, builds the analysis context, and generates the final investment analysis response. The result is also saved to the configured JSONL log file.


---

## 🔐 API Key Setup (Required)

This notebook uses **Google Colab Secrets** to securely manage API keys.

Before running the notebook, you must register your API key:

### Steps
1. Open the sidebar in Colab
2. Click the 🔑 **Secrets** tab
3. Click **+ New secret**
4. Add the following:

- **Name**: `OPENAI_API_KEY`  
- **Value**: Your API key

⚠️ The notebook will not run without this step.

---


How to use this notebook:
1. In the next code cell, load `OPENAI_API_KEY` from Colab Secrets and review the default settings.
2. Run the setup cell to clone the repository and install dependencies.
3. Run the environment setup cell to configure environment variables for the current session.
4. Optionally review the available personas.
5. In the query cell near the bottom, set `QUERY` and optionally `PERSONA_NAME`.
6. Run the sample query cell and display results as HTML.
7. Check the logs.
8. Launch the Interactive Wallstreet-AI UI.

This notebook will:
- Clone the repository if it is not already present
- Install the required packages
- Configure environment variables for the current session
- Run `pipeline()`
- Print the analysis result and let you inspect the latest saved log entry

## Step 1. Update your API key and default settings

In [ ]:
from google.colab import userdata

GITHUB_REPO_URL = "https://github.com/davidkim205/wallstreet-ai.git"

LLM_MODEL_API_KEY = userdata.get('OPENAI_API_KEY')
LLM_MODEL_NAME = "gpt-5-mini"

# Output file names
LOG_FILE = "analysis_results.jsonl"
PERSONA_FILE = "persona.jsonl"

## Step 2. Clone the repository and install dependencies

Run the next cell as-is. It checks whether the repository already exists in Colab, clones it if needed, installs the required packages, and moves into the project directory.


In [3]:
import os
import subprocess
from pathlib import Path

REPO_DIR = Path("./content/wallstreet-ai")

if not LLM_MODEL_API_KEY:
    raise ValueError("Please set LLM_MODEL_API_KEY before running this notebook.")

if REPO_DIR.exists():
    print(f"Repository already exists: {REPO_DIR}")
else:
    result = subprocess.run(
        ["git", "clone", GITHUB_REPO_URL, str(REPO_DIR)],
        text=True,
        capture_output=True,
    )

    print("returncode:", result.returncode)
    print("stdout:", result.stdout)
    print("stderr:", result.stderr)

    result.check_returncode()

subprocess.run(["python", "-m", "pip", "install", "-r", str(REPO_DIR / "requirements.txt")], check=True)

os.chdir(REPO_DIR)
print("Current working directory:", Path.cwd())

returncode: 0
stdout: 
stderr: 'content/wallstreet-ai'에 복제합니다...

Current working directory: /work/jupyter/content/wallstreet-ai


## Step 3. Write environment variables

The next cell stores the values you entered above in both the current Python session and a local `.env` file inside the repository.


In [4]:
os.environ["LLM_MODEL_API_KEY"] = LLM_MODEL_API_KEY
os.environ["LLM_MODEL_NAME"] = LLM_MODEL_NAME
os.environ["LOG_FILE"] = LOG_FILE
os.environ["PERSONA_FILE"] = PERSONA_FILE

print("Environment variables configured for the current session.")

Environment variables configured for the current session.


## Step 4. Review available personas

If you want the analysis to reflect a specific investing style, run the next cell and copy one of the printed names into `PERSONA_NAME` in the query cell below. If you prefer the default behavior, keep `PERSONA_NAME = None`.


In [5]:
from persona.persona_loader import load_personas

personas = load_personas()

print(f"Loaded personas: {len(personas)}")
for idx, persona in enumerate(personas, start=1):
    print(f"{idx}. {persona.name}")

Loaded personas: 5
1. J.P. 모건
2. 워런 버핏
3. 짐 로저스
4. 켄 그리핀
5. 레이 달리오


## Step 5. Set the query and optional persona

Edit the next cell with the question you want to analyze. If you want to apply a persona, copy one of the names printed above into `PERSONA_NAME`. Otherwise, keep `PERSONA_NAME = None`.


In [6]:
# Set the query to run and the persona name to use.
# If you do not want to use a persona, keep PERSONA_NAME = None.
QUERY = "2025년 4분기 삼성전자 실적은 어땠나요?"
# PERSONA_NAME = "J.P. 모건"
PERSONA_NAME = None

## Step 6. Run Sample Query and Review HTML Output Examples

Below, we will run example query and display its analysis result in a separate HTML block.

In [7]:
# @title
from IPython.display import HTML, display
import pipeline as pipeline_module
import importlib

# Common HTML styles
HTML_STYLE = """
<style>
    body {
        font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
        line-height: 1.8;
        color: #333;
        background-color: #f4f7f6;
        margin: 0;
        padding: 0;
    }
    .container {
        width: 90%;
        max-width: 1000px;
        margin: 30px auto;
        background: #ffffff;
        padding: 30px;
        border-radius: 12px;
        box-shadow: 0 6px 20px rgba(0,0,0,0.08);
    }
    h1 {
        color: #1a237e; /* Deep blue */
        text-align: center;
        margin-bottom: 40px;
        font-size: 2.5em;
        font-weight: 600;
        border-bottom: 3px solid #3f51b5; /* Indigo */
        padding-bottom: 15px;
    }
    h2 {
        color: #3f51b5; /* Indigo */
        border-bottom: 2px solid #9fa8da; /* Lighter indigo */
        padding-bottom: 10px;
        margin-top: 40px;
        font-size: 1.8em;
    }
    h3 {
        color: #5c6bc0; /* Medium indigo */
        font-size: 1.3em;
        margin-top: 25px;
        margin-bottom: 15px;
    }
    .query-section {
        background: #e8eaf6; /* Light indigo background */
        padding: 25px;
        border-radius: 10px;
        margin-bottom: 30px;
        border: 1px solid #c5cae9;
    }
    .query-text-inline { /* New style for inline query text */
        font-weight: 600;
        color: #2c3e50; /* Changed from red to deep charcoal blue for emphasis */
        font-size: 1.1em;
        display: inline;
    }
    .analysis-result {
        background: #fcfcfc;
        border-left: 6px solid #42a5f5; /* Light blue */
        padding: 20px;
        margin-top: 15px;
        white-space: pre-wrap;
        word-wrap: break-word;
        overflow-wrap: anywhere;
        height: 380px;
        box-sizing: border-box;
        overflow-y: auto;
        overflow-x: hidden;
        border-radius: 6px;
        box-shadow: inset 0 1px 5px rgba(0,0,0,0.05);
    }
</style>
"""

def display_single_result_html(query_text, analysis_result):
    analysis_text = analysis_result.llm_response if hasattr(analysis_result, 'llm_response') else str(analysis_result)
    single_html_output = f"""
    <div class="container">
        <div class="query-section">
            <h2>Query: <span class="query-text-inline">{query_text}</span></h2>
            <h3>Analysis:</h3>
            <div class="analysis-result">{analysis_text}</div>
        </div>
    </div>
    """
    display(HTML(HTML_STYLE + single_html_output))

print("HTML display function loaded. Ready to run individual queries.")

HTML display function loaded. Ready to run individual queries.


In [8]:
# Reload pipeline module to ensure any changes are picked up
importlib.reload(pipeline_module)

result = pipeline_module.pipeline(
    QUERY,
    persona_name=PERSONA_NAME,
    stream=False,
)


Wallstreet-AI 분석 시작: 2025년 4분기 삼성전자 실적은 어땠나요?
[②] Tool Router → 선택된 도구: ['earnings', 'fundamentals', 'news', 'web_search']
[③] 데이터 수집 중 (ticker=005930.KS, period=1y)...
    → 펀더멘털: 52개 지표
    → 뉴스: 8개 헤드라인


/home/ai/anaconda3/envs/jupyter/lib/python3.11/site-packages/yfinance/scrapers/fundamentals.py:36: DeprecationWarning: 'Ticker.earnings' is deprecated as not available via API. Look for "Net Income" in Ticker.income_stmt.
  warnings.warn("'Ticker.earnings' is deprecated as not available via API. Look for \"Net Income\" in Ticker.income_stmt.", DeprecationWarning)


    → 분기 실적: 6개 / 연간 실적: 4개 / EPS 서프라이즈: 0개
[INFO] - 삼성전자 2025년 4분기 실적 검색 중...


[INFO] 기사 수집 진행: 3/3 [00:02<00:00]


[INFO] - 삼성전자 2025년 4분기 영업이익 검색 중...


...

[INFO] - 삼성전자 반도체 2025년 4분기 실적 검색 중...


[INFO] 기사 수집 진행: 3/3 [00:02<00:00]


[INFO] - 삼성전자 2025년 4분기 컨퍼런스콜 검색 중...


[INFO] 기사 수집 진행: 3/3 [00:03<00:00]


[⑤] 단일 응답 생성 중...

[완료] 분석 완료 ✓

[소요시간(sec)]:
  intent_parse        : 4.85s
  tool_route          : 0.00s
  data_collect        : 2.10s
  context_build       : 25.98s
  analysis_generate   : 50.30s
  total               : 83.22s


In [ ]:
display_single_result_html(QUERY, result)

## Step 7. How to check logs

Analysis results are saved to the specified log file. You can run the code below to check the location of the log file, or activate the commented-out code to view the latest log entry directly.

In [10]:
import json

log_file = os.environ.get("LOG_FILE")
if not log_file:
    print("LOG_FILE environment variable is not set. Skipping log preview.")
else:
    log_path = Path(log_file)
    if not log_path.exists():
        print(f"No log file yet: {log_path}")
        print("Run the pipeline cell above first to generate an analysis log.")
    else:
        lines = [line for line in log_path.read_text(encoding="utf-8").splitlines() if line.strip()]
        if not lines:
            print(f"The log file is empty: {log_path}")
        else:
            print(f"Showing the most recent log entry from: {log_path}\n")
            print(json.dumps(json.loads(lines[-1]), ensure_ascii=False, indent=2))

Showing the most recent log entry from: analysis_results.jsonl

{
  "timestamp": "2026-04-02 13:18:41",
  "query": "2025년 4분기 삼성전자 실적은 어땠나요?",
  "ticker": "005930.KS",
  "analysis_type": "earnings",
  "data_context": {
    "ticker": "005930.KS",
    "price_data": {},
    "fundamentals": {
      "company_name": "Samsung Electronics Co., Ltd.",
      "symbol": "005930.KS",
      "exchange": "KSC",
      "quote_type": "EQUITY",
      "currency": "KRW",
      "sector": "Technology",
      "industry": "Consumer Electronics",
      "country": "South Korea",
      "city": "Suwon-si",
      "website": "https://www.samsung.com",
      "full_time_employees": null,
      "market_cap_b": 1190791.36,
      "enterprise_value_b": 1168600.74,
      "shares_outstanding_b": 5.88,
      "float_shares_b": 5.46,
      "pe_ratio": null,
      "forward_pe": 6.1996183,
      "pb_ratio": null,
      "ps_ratio": 3.569455,
      "ev_to_ebitda": 13.092,
      "trailing_eps": null,
      "forward_eps": null,
     

## Step 8. Launch Interactive Wallstreet-AI UI

Run the cell below to launch the Wallstreet-AI interactive UI directly in the output area of this cell. You can enter your query, click 'Run Analysis' to get the analysis result, and use the 'Clear' button to clear the chat history.

In [ ]:
# @title
import builtins
import contextlib
import importlib
import html
import io
import logging
import ipywidgets as widgets
from IPython.display import HTML, display

importlib.reload(pipeline_module)

display(HTML("""
<style>
  .ws-basic-section {
    margin-top: 14px;
    margin-bottom: 8px;
    font-size: 12px;
    font-weight: 800;
    letter-spacing: 0.06em;
    text-transform: uppercase;
    color: #47607c;
  }
  .ws-surface-panel {
    border-radius: 14px !important;
    border: 1px solid #bfdbfe !important;
    background: linear-gradient(180deg, #ffffff 0%, #f8fbff 100%) !important;
    box-sizing: border-box !important;
    box-shadow: inset 0 1px 0 rgba(255, 255, 255, 0.8), 0 8px 20px rgba(148, 163, 184, 0.10) !important;
  }
  .ws-column-panel {
    border-radius: 16px !important;
    border: 1px solid #bfdbfe !important;
    background: linear-gradient(180deg, #ffffff 0%, #f8fbff 100%) !important;
    box-sizing: border-box !important;
    height: 100% !important;
    overflow-x: hidden !important;
    box-shadow: inset 0 1px 0 rgba(255, 255, 255, 0.82), 0 10px 24px rgba(148, 163, 184, 0.12) !important;
  }
  .ws-right-column,
  .ws-right-column > div,
  .ws-right-column .widget-box {
    max-width: 100% !important;
    overflow-x: hidden !important;
    box-sizing: border-box !important;
  }
  .ws-result-output textarea,
  .ws-log-output textarea {
    width: calc(100% - 2px) !important;
    max-width: 100% !important;
    box-sizing: border-box !important;
    overflow-y: auto !important;
    overflow-x: hidden !important;
    white-space: pre-wrap !important;
    word-break: break-word !important;
    overflow-wrap: anywhere !important;
    resize: none !important;
    padding: 12px !important;
    border: none !important;
    background: transparent !important;
  }
  .ws-result-output textarea {
    height: 492px !important;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif !important;
    font-size: 13px !important;
    line-height: 1.35 !important;
    color: #0f172a !important;
  }
  .ws-log-output textarea {
    height: 176px !important;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif !important;
    font-size: 12px !important;
    line-height: 1.12 !important;
    color: #0f172a !important;
  }
  .ws-primary-button button,
  .ws-secondary-button button {
    border-radius: 14px !important;
    border: 1px solid #bfdbfe !important;
    font-weight: 800 !important;
    font-size: 13px !important;
    letter-spacing: 0.01em !important;
    padding: 0 16px !important;
    transition: all 0.18s ease !important;
    background: linear-gradient(180deg, #ffffff 0%, #f8fbff 100%) !important;
    box-shadow: inset 0 1px 0 rgba(255, 255, 255, 0.8), 0 8px 20px rgba(148, 163, 184, 0.10) !important;
  }
  .ws-primary-button button {
    color: #1d4ed8 !important;
    border-color: #93c5fd !important;
  }
  .ws-primary-button button:hover {
    background: linear-gradient(180deg, #eff6ff 0%, #dbeafe 100%) !important;
    border-color: #60a5fa !important;
    transform: translateY(-1px);
    box-shadow: 0 0 0 4px rgba(96, 165, 250, 0.14), 0 12px 24px rgba(37, 99, 235, 0.10) !important;
  }
  .ws-secondary-button button {
    color: #475569 !important;
    border-color: #cbd5e1 !important;
  }
  .ws-secondary-button button:hover {
    background: linear-gradient(180deg, #ffffff 0%, #f1f5f9 100%) !important;
    border-color: #94a3b8 !important;
    transform: translateY(-1px);
    box-shadow: 0 10px 22px rgba(148, 163, 184, 0.14) !important;
  }
  .ws-primary-button button:disabled,
  .ws-secondary-button button:disabled {
    opacity: 0.6 !important;
    box-shadow: none !important;
    transform: none !important;
  }
  .ws-input-field select,
  .ws-input-field textarea,
  .ws-input-field input {
    border-radius: 14px !important;
    border: 1px solid #bfdbfe !important;
    background: linear-gradient(180deg, #ffffff 0%, #f8fbff 100%) !important;
    color: #0f172a !important;
    box-shadow: inset 0 1px 0 rgba(255, 255, 255, 0.8), 0 8px 20px rgba(148, 163, 184, 0.10) !important;
    transition: border-color 0.18s ease, box-shadow 0.18s ease, background 0.18s ease !important;
    padding: 10px 12px !important;
  }
  .ws-input-field select {
    min-height: 42px !important;
  }
  .ws-input-field textarea {
    line-height: 1.5 !important;
    resize: vertical !important;
  }
  .ws-input-field select:focus,
  .ws-input-field textarea:focus,
  .ws-input-field input:focus {
    border-color: #60a5fa !important;
    box-shadow: 0 0 0 4px rgba(96, 165, 250, 0.18), 0 10px 24px rgba(37, 99, 235, 0.12) !important;
    outline: none !important;
    background: #ffffff !important;
  }
  .ws-mono-note,
  .ws-mono-note select,
  .ws-mono-note textarea,
  .ws-mono-note input,
  .ws-mono-note option {
    font-family: "SFMono-Regular", Consolas, "Liberation Mono", Menlo, monospace !important;
  }
</style>
"""))

persona_names = [""]
if "personas" in globals():
    persona_names.extend([persona.name for persona in personas])

persona_title = widgets.HTML("<div class=\"ws-basic-section\">Persona</div>")
persona_dropdown = widgets.Dropdown(
    options=[("Persona 선택 안 함", "")] + [(name, name) for name in persona_names if name],
    value=PERSONA_NAME or "",
    description="",
    layout=widgets.Layout(width="100%", margin="0 0 12px 0"),
)
persona_dropdown.add_class("ws-input-field")
persona_dropdown.add_class("ws-mono-note")

query_title = widgets.HTML("<div class=\"ws-basic-section\">Query</div>")
query_input = widgets.Textarea(
    value=QUERY,
    placeholder="질문을 입력하세요",
    description="",
    layout=widgets.Layout(width="100%", height="88px", margin="0 0 12px 0"),
)
query_input.add_class("ws-input-field")
query_input.add_class("ws-mono-note")

ask_button = widgets.Button(
    description="Run Analysis",
    button_style="",
    layout=widgets.Layout(width="144px", height="38px"),
)
ask_button.add_class("ws-primary-button")
clear_button = widgets.Button(
    description="Clear",
    button_style="",
    layout=widgets.Layout(width="144px", height="38px"),
)
clear_button.add_class("ws-secondary-button")
button_group = widgets.HBox(
    [ask_button, clear_button],
    layout=widgets.Layout(gap="10px", justify_content="flex-end", flex="0 0 auto"),
)

status_output = widgets.Output(layout=widgets.Layout(flex="0 0 auto", width="auto", margin="0"))
action_row = widgets.HBox(
    [button_group],
    layout=widgets.Layout(width="100%", justify_content="flex-end", margin="4px 0 16px 0"),
)

log_title = widgets.HTML("<div class=\"ws-basic-section\">Run Log</div>")
log_header_row = widgets.HBox(
    [log_title, status_output],
    layout=widgets.Layout(width="100%", align_items="center", justify_content="space-between", gap="12px"),
)
log_output = widgets.Textarea(
    value="",
    description="",
    disabled=True,
    layout=widgets.Layout(width="100%", height="176px", margin="0 0 16px 0", overflow="hidden", border="1px solid #d7deea", padding="0", border_radius="14px", box_sizing="border-box"),
)
log_output.add_class("ws-surface-panel")
log_output.add_class("ws-log-output")

output_title = widgets.HTML("<div class=\"ws-basic-section\">Results</div>")
result_output = widgets.Textarea(
    value="",
    description="",
    disabled=True,
    layout=widgets.Layout(width="calc(100% - 4px)", height="492px", min_height="492px", overflow="hidden", border="1px solid #d7deea", padding="0", border_radius="14px", box_sizing="border-box"),
)
result_output.add_class("ws-surface-panel")
result_output.add_class("ws-result-output")

log_state = {"text": ""}
raw_log_state = {"text": ""}
result_state = {"text": ""}
log_filter_state = {"suppress_stdout_after_phase5": False}
EMPTY_LOG_MESSAGE = "No logs were captured."

def render_log_output():
    log_text = log_state["text"]
    log_output.value = log_text

def render_result_output():
    result_text = result_state["text"].strip()
    result_output.value = result_state["text"] if result_text else ""

def append_result_delta(delta):
    if delta is None:
        return
    delta = str(delta)
    if not delta:
        return
    result_state["text"] = f"{result_state['text']}{delta}"
    result_output.value = result_state["text"]

def append_raw_log(text):
    if text is None:
        return
    text = str(text)
    if not text:
        return
    raw_log_state["text"] = f"{raw_log_state['text']}{text}"

def append_event_log(event_type, message):
    if message is None:
        return
    message = str(message)
    if not message:
        return
    if event_type == "stdout" and not message.strip():
        return
    prefix = f"[{event_type}] "
    normalized = message.replace("\r\n", "\n")
    if normalized.endswith("\n"):
        formatted = "".join(prefix + line + "\n" for line in normalized.rstrip("\n").split("\n"))
    else:
        formatted = "".join(prefix + line + "\n" for line in normalized.split("\n"))
    log_state["text"] = f"{log_state['text']}{formatted}"
    log_output.value = log_state["text"]

render_log_output()
render_result_output()

def reset_outputs():
    log_state["text"] = ""
    raw_log_state["text"] = ""
    result_state["text"] = ""
    log_filter_state["suppress_stdout_after_phase5"] = False
    render_log_output()
    render_result_output()

def show_status(message, background, foreground, border):
    status_output.clear_output()
    with status_output:
        display(HTML(
            f"<div style=\"display:inline-flex;align-items:center;gap:8px;padding:5px 10px;border-radius:999px;background:{background};color:{foreground};font-size:12px;font-weight:700;letter-spacing:0.01em;border:1px solid {border};white-space:nowrap;box-shadow:0 1px 2px rgba(15,23,42,0.04);\"><span style=\"width:7px;height:7px;border-radius:999px;background:{foreground};display:inline-block;opacity:0.9;\"></span>{message}</div>"
        ))

STATUS_LABELS = {
    "인텐트 분석 중...": "Analyzing intent...",
    "도구 라우팅 중...": "Routing tools...",
    "시장 데이터 수집 중...": "Collecting market data...",
    "뉴스 컨텍스트 생성 중...": "Building news context...",
    "컨텍스트 빌드 중...": "Building context...",
    "LLM 분석 생성 중...": "Generating analysis...",
}

def format_status_label(message):
    return STATUS_LABELS.get(message, message)

def handle_status_event(message):
    show_status(format_status_label(message), "#dbeafe", "#1d4ed8", "#93c5fd")
    append_event_log("status", message)

def flush_captured_output(stdout_buffer, stderr_buffer):
    captured_stdout = stdout_buffer.getvalue()
    captured_stderr = stderr_buffer.getvalue()
    if captured_stdout.strip() and captured_stdout not in raw_log_state["text"]:
        append_raw_log(captured_stdout)
    if captured_stderr.strip():
        append_raw_log("\n[stderr]\n" + captured_stderr)
        append_event_log("stderr", captured_stderr)
    if not log_state["text"].strip():
        log_state["text"] = f"[done] {EMPTY_LOG_MESSAGE}\n"
        render_log_output()

def append_log(text):
    if text is None:
        return
    text = str(text)
    if not text:
        return
    append_raw_log(text)
    if not log_filter_state["suppress_stdout_after_phase5"]:
        append_event_log("stdout", text)
    if "[⑤]" in text:
        log_filter_state["suppress_stdout_after_phase5"] = True

def make_widget_print(original_print):
    def widget_print(*args, **kwargs):
        sep = kwargs.get("sep", " ")
        end = kwargs.get("end", "\n")
        text = sep.join(str(arg) for arg in args) + end
        append_log(text)
        return original_print(*args, **kwargs)
    return widget_print

class WidgetLogHandler(logging.Handler):
    def emit(self, record):
        append_raw_log(f"{self.format(record)}\n")
        append_event_log("stdout", self.format(record))

@contextlib.contextmanager
def capture_named_loggers(logger_names, level=logging.WARNING):
    handler = WidgetLogHandler()
    handler.setLevel(level)
    handler.setFormatter(logging.Formatter("%(levelname)s:%(name)s:%(message)s"))
    previous_states = []
    try:
        for logger_name in logger_names:
            logger = logging.getLogger(logger_name)
            previous_states.append((logger, list(logger.handlers), logger.level, logger.propagate, logger.disabled))
            logger.handlers = [handler]
            logger.setLevel(level)
            logger.propagate = False
            logger.disabled = False
        yield
    finally:
        for logger, handlers, logger_level, propagate, disabled in previous_states:
            logger.handlers = handlers
            logger.setLevel(logger_level)
            logger.propagate = propagate
            logger.disabled = disabled

def run_basic_ui(_):
    query = query_input.value.strip()
    persona_name = persona_dropdown.value or None

    status_output.clear_output()
    reset_outputs()

    if not query:
        with status_output:
            display(HTML("<div style=\"color:#9a3412;font-weight:700;\">Please enter a query first.</div>"))
        return

    ask_button.disabled = True
    clear_button.disabled = True

    stdout_buffer = io.StringIO()
    stderr_buffer = io.StringIO()
    original_print = builtins.print

    try:
        show_status("Running analysis...", "#dbeafe", "#1d4ed8", "#93c5fd")
        append_event_log("status", "Running analysis...")

        append_event_log("stdout", "[UI] Starting pipeline run.")
        importlib.reload(pipeline_module)

        builtins.print = make_widget_print(original_print)
        with capture_named_loggers(["trafilatura", "trafilatura.core", "trafilatura.utils"]):
            with contextlib.redirect_stdout(stdout_buffer), contextlib.redirect_stderr(stderr_buffer):
                result = pipeline_module.pipeline(
                    query,
                    persona_name=persona_name,
                    status_callback=handle_status_event,
                    stream_callback=append_result_delta,
                    stream=True,
                )

        flush_captured_output(stdout_buffer, stderr_buffer)

        if not result_state["text"].strip():
            result_state["text"] = getattr(result, "llm_response", None) or ""
            render_result_output()
        append_event_log("result", f"query={result.query}, ticker={result.ticker}, analysis_type={result.analysis_type}")
        append_event_log("done", "Analysis stream finished.")

        show_status("Analysis complete.", "#dcfce7", "#166534", "#86efac")
    except Exception as exc:
        flush_captured_output(stdout_buffer, stderr_buffer)
        if not result_state["text"].strip():
            result_state["text"] = f"Error: {exc}"
        else:
            result_state["text"] = f"{result_state['text']}\n\nError: {exc}"
        render_result_output()
        append_event_log("error", str(exc))
        show_status("An error occurred while running the analysis.", "#fee2e2", "#b91c1c", "#fca5a5")
    finally:
        builtins.print = original_print
        ask_button.disabled = False
        clear_button.disabled = False

def clear_basic_ui(_):
    query_input.value = ""
    persona_dropdown.value = ""
    reset_outputs()
    status_output.clear_output()

left_content = widgets.VBox([
    persona_title,
    persona_dropdown,
    query_title,
    query_input,
    action_row,
    log_header_row,
    log_output,
], layout=widgets.Layout(width="100%", height="100%"))

left_column = widgets.Box([
    left_content,
], layout=widgets.Layout(width="auto", flex="1 1 0", min_width="420px", min_height="560px", height="100%", padding="18px"))
left_column.add_class("ws-column-panel")
left_column.add_class("ws-left-column")

right_content = widgets.VBox([
    output_title,
    result_output,
], layout=widgets.Layout(width="100%", min_width="0", height="100%", flex="1 1 auto", display="flex", flex_flow="column", align_items="stretch"))

right_column = widgets.Box([
    right_content,
], layout=widgets.Layout(width="auto", flex="1 1 0", min_width="360px", min_height="560px", height="100%", padding="18px", display="flex", flex_flow="column", align_items="stretch"))
right_column.add_class("ws-column-panel")
right_column.add_class("ws-right-column")

height_sync = widgets.HTML("""
<script>
(function(){
  function syncHeights(){
    const left = document.querySelector('.ws-left-column');
    const right = document.querySelector('.ws-right-column');
    if (!left || !right) return;
    left.style.minHeight = '';
    right.style.minHeight = '';
    const target = Math.max(left.offsetHeight, right.offsetHeight);
    left.style.minHeight = target + 'px';
    right.style.minHeight = target + 'px';
  }
  requestAnimationFrame(syncHeights);
  window.addEventListener('resize', syncHeights);
  const observer = new ResizeObserver(() => syncHeights());
  requestAnimationFrame(() => {
    const left = document.querySelector('.ws-left-column');
    const right = document.querySelector('.ws-right-column');
    if (left) observer.observe(left);
    if (right) observer.observe(right);
  });
})();
</script>
""")

scroll_sync = widgets.HTML("""
<script>
(function(){
  function scrollLogToBottom(){
    const textarea = document.querySelector('.ws-log-output textarea');
    if (!textarea) return;
    textarea.scrollTop = textarea.scrollHeight;
  }

  function scrollResultsToBottom(){
    const textarea = document.querySelector('.ws-result-output textarea');
    if (!textarea) return;
    textarea.scrollTop = textarea.scrollHeight;
  }

  function attachObservers(){
    const logTextarea = document.querySelector('.ws-log-output textarea');
    const resultTextarea = document.querySelector('.ws-result-output textarea');

    if (logTextarea && !logTextarea.dataset.scrollBound) {
      logTextarea.dataset.scrollBound = '1';
      let prev = logTextarea.value;
      setInterval(() => {
        const el = document.querySelector('.ws-log-output textarea');
        if (!el) return;
        if (el.value !== prev) {
          prev = el.value;
          scrollLogToBottom();
        }
      }, 120);
    }

    if (resultTextarea && !resultTextarea.dataset.scrollBound) {
      resultTextarea.dataset.scrollBound = '1';
      let prev = resultTextarea.value;
      setInterval(() => {
        const el = document.querySelector('.ws-result-output textarea');
        if (!el) return;
        if (el.value !== prev) {
          prev = el.value;
          scrollResultsToBottom();
        }
      }, 120);
    }

    scrollLogToBottom();
    scrollResultsToBottom();
  }

  requestAnimationFrame(attachObservers);
  setTimeout(attachObservers, 150);
  setTimeout(attachObservers, 600);
  window.addEventListener('resize', () => {
    scrollLogToBottom();
    scrollResultsToBottom();
  });
})();
</script>
""")

app = widgets.HBox([
    left_column,
    right_column,
], layout=widgets.Layout(width="100%", max_width="1100px", margin="0 auto", align_items="stretch", justify_content="space-between", gap="14px"))
app_shell = widgets.VBox([app, height_sync, scroll_sync], layout=widgets.Layout(width="100%", max_width="1100px", margin="0 auto"))

ask_button.on_click(run_basic_ui)
clear_button.on_click(clear_basic_ui)


display(app_shell)
